# Scenario 5: Claude Code for Continuous Integration

## 1. In One Minute

**What it is.** Claude Code runs inside CI for automated code review, test generation, and PR feedback. The goal is useful findings with few false positives.

**Domains hit:** D3 Claude Code Configuration and Workflows, D4 Prompt Engineering and Structured Output.

**Task statements:** TS 3.1 (CLAUDE.md for CI context), 3.6 (CI/CD), 4.1 (explicit criteria), 4.2 (few-shot), 4.5 (Batches API), 4.6 (multi-pass review).

**Top 3 traps**
1. Forgetting `-p` (print mode), so the job waits for input and hangs.
2. Using the Message Batches API for a blocking pre-merge check.
3. Reviewing a big PR in one pass, or telling the model to "be conservative" instead of giving explicit criteria.

## 2. Architecture Diagram

```mermaid
flowchart TD
    PR["Pull request opened"] --> CI["CI job:<br>claude -p ... --output-format json<br>--json-schema"]
    CI --> Files["Pass 1: per-file review<br>(local issues)"]
    Files --> Integ["Pass 2: cross-file<br>integration review"]
    Integ --> Dedupe["Drop findings already<br>reported on earlier runs"]
    Dedupe --> Post["Post structured comments<br>on the PR"]
    Post --> Gate{"Blocking severity?"}
    Gate -->|"Yes"| Fail["Exit code 1: block merge"]
    Gate -->|"No"| Pass["Exit code 0"]
    Nightly["Overnight tech-debt report"] --> Batch["Message Batches API<br>(cheaper, up to 24h)"]
```

## 3. How a Good Answer Thinks

1. **Is a human waiting?** Blocking pre-merge check: real-time API. Overnight or weekly report: Message Batches API (50% cheaper, up to 24h, no latency guarantee).
2. **Non-interactive?** Use `-p` so the job never waits for input. Add `--output-format json` and `--json-schema` so the next CI step can parse the result.
3. **False positives?** Write explicit criteria: what to report, what to skip, how severe. "Be conservative" is vague.
4. **Large PR?** Per-file passes for local issues, then one integration pass for cross-file problems.
5. **Independent review.** The session that wrote the code is biased toward it. Review in a separate instance.
6. **Repeated runs.** Pass prior findings and ask for only new or unaddressed issues.
7. **Context for CI.** Put testing standards and fixtures in `CLAUDE.md` and give existing tests in context.

### Model and Sampling Notes
- Use `claude-sonnet-4-5` for coordinators and reasoning, `claude-haiku-4-5` for workers and extraction.
- With extended thinking on: omit `temperature`, leave `top_k` and `top_p` unset, and use only `tool_choice` `auto` or `none` (forced or `any` returns HTTP 400). Forced-tool extraction calls must not use thinking.
- In tool loops, return thinking blocks with their signature unchanged.
- Loops end on `stop_reason` only (`end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `pause_turn`, `refusal`). A max-iterations counter is a safety cap, not the stop rule.

## 4. Task Statement Map

| Task statement | How it shows up here | Trap |
|---|---|---|
| TS 3.1 CLAUDE.md | Project test and review standards for CI | Repeating them in every prompt |
| TS 3.6 CI/CD | `claude -p ... --output-format json --json-schema ...` | Interactive mode in CI |
| TS 4.1 Explicit criteria | Report and skip lists with severity | "Be conservative", confidence filters |
| TS 4.2 Few-shot | Examples of real issue vs acceptable pattern | Long rule lists with no examples |
| TS 4.5 Batches | Overnight report only | Batching the blocking check |
| TS 4.6 Multi-pass | Per-file passes then integration pass | One pass over many files |

## 5. Worked Example

**Setup:** A 14-file PR. The pre-merge check must finish quickly. A nightly tech-debt report also exists.

1. CI runs `claude -p "Review this PR" --output-format json --json-schema '<schema>'`. No prompt, no hang.
2. Pass 1 reviews each file alone: bare `except`, hardcoded keys outside tests.
3. Pass 2 reviews all changed files together: the client sends `auth_token`, the API expects `bearer_token`.
4. A prior run already flagged a bare `except`. The new run reports only new items; the earlier issue still counts for the merge decision.
5. Any CRITICAL or HIGH issue returns exit code 1 and blocks merge.
6. The nightly tech-debt report goes through the Batches API.

Try it offline: `scenarios/scenario_5_continuous_integration/simulation.py` (`build_ci_command`, `filter_new_findings`, `run_two_pass_review`).

## 7. Self-Check Checklist

- [ ] I know why `-p` is needed and what it prevents.
- [ ] I can say which workload uses Batches and which does not.
- [ ] I can write report and skip criteria for a review prompt.
- [ ] I can explain per-file plus integration passes.
- [ ] I handle repeated runs by passing prior findings.

**Memory hook:** "-p to not hang, schema to parse, batch the sleepers, review in two passes."


In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_5_continuous_integration.simulation import *

print(' Scenario Workbench Initialized: Scenario 5: Continuous Integration Pipeline Reviewer')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Test Workload API Routing (Synchronous PR vs Message Batches)
engine = ContinuousIntegrationEngine(headless_flag_present=True)

api_pr = engine.route_api_for_workload(WorkloadType.BLOCKING_PR_CHECK)
api_audit = engine.route_api_for_workload(WorkloadType.OVERNIGHT_REPO_AUDIT)
print("Blocking PR API:", api_pr)
print("Overnight Audit API:", api_audit)
assert api_pr == "REALTIME_MESSAGES"
assert api_audit == "MESSAGE_BATCHES"



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Test Two-Pass Review Pipeline & Negative Boundary Filtering
engine = ContinuousIntegrationEngine(headless_flag_present=True)
pr_files = {
    "api/routes.py": "def handle(bearer_token: str): pass",
    "client/sdk.py": "def call(): return handle(auth_token='abc')",
    "tests/test_app.py": "mock_secret = 'sk-ant-live_fake_in_tests'"
}
report = engine.run_two_pass_review(pr_files)
print("CI Verdict:", report.status)
print("Exit Code:", report.exit_code)
assert report.exit_code == 1
assert report.status == "CHANGES_REQUESTED"
assert any(f.pass_number == 2 for f in report.findings)
assert not any("mock_secret" in f.description for f in report.findings)



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.


### Step 5: New behaviour - CI command builder and prior-findings dedupe
Offline checks for the extended simulation.


In [ ]:
cmd = build_ci_command("Review this PR")
assert cmd[1] == "-p" and "--json-schema" in cmd
engine = ContinuousIntegrationEngine()
files = {"src/a.py": "try:\n    x()\nexcept:\n    pass\n"}
first = engine.run_two_pass_review(files)
second = engine.run_two_pass_review(files, prior_findings=list(first.findings))
assert second.new_findings == [] and second.exit_code == 1
print("Command:", cmd[:4])
